# 3 · Build your visual EEG review tool
 **Goal:** use the model and evidence you created, without writing a web application.
 A few supplied Python calls provide selectable validation examples, class-specific heatmaps,
 replacement sensitivity, nearest training images, quality flags, and an auditable review record.
 Run Notebook 2 first in the same filesystem or copy its complete output directory into this runtime.

## Before running
Use Python 3.11 or 3.12 in Colab or local Jupyter. Code is provided: **predict → run → inspect → explain**.
Downloads and training happen only when **you** run the relevant cells. This source notebook has no precomputed results.
Use the [PhysioNet release](https://physionet.org/content/eegmmidb/1.0.0/) under its Open Data Commons Attribution License v1.0.
Cite Schalk (2009), DOI 10.13026/C28G6P, Schalk et al. (2004), DOI 10.1109/TBME.2004.827072,
and the platform citation listed on that release. Data remain outside Git.
The helper is downloaded from this course repository; inspect/review its source before running.
The default moving `main` makes starting easy; pin `SOURCE_REF` to a reviewed commit for a durable submission.
Save the exact helper bytes/hash, source notebook, package versions, configuration, and outputs.

In [ ]:
%pip -q install "numpy>=1.26,<3" "scipy>=1.11,<2" "mne>=1.7,<2" "matplotlib>=3.8,<4" "pandas>=2,<3" "Pillow>=10,<13" "scikit-learn>=1.4,<2" "ipywidgets>=8,<9"

In [ ]:
%pip -q install "tensorflow>=2.16,<3"

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, urllib.request
SOURCE_REF = "main"  # For a repeatable handoff, replace with a reviewed student-repository commit SHA.
helper = Path("chapter18_vision_studio.py")
url = ("https://raw.githubusercontent.com/ZagElSayed/Introduction-to-Neuroinformatics/"
       + SOURCE_REF + "/chapters/ch18-vision-ai/vision_studio.py")
payload = urllib.request.urlopen(url, timeout=60).read()
helper.write_bytes(payload)  # Keep this exact helper with your evidence; do not rely on a moving branch.
HELPER_HASH = hashlib.sha256(payload).hexdigest()
spec = importlib.util.spec_from_file_location("chapter18_vision_studio", helper)
studio = importlib.util.module_from_spec(spec)
spec.loader.exec_module(studio)
print("Helper SHA-256:", HELPER_HASH)
print(studio.versions())

In [ ]:
DATA_DIR = Path("outputs/ch18_studio/REPLACE_WITH_YOUR_RUN_TAG")
if not (DATA_DIR / "model.keras").exists():
    raise FileNotFoundError("Set DATA_DIR to the output directory printed by Notebook 2.")
# Prefer the exact helper saved with the model over a potentially newer repository helper.
source_file = DATA_DIR / "vision_studio_source.py"
saved_source = json.loads((DATA_DIR / "helper_source.json").read_text())
if hashlib.sha256(source_file.read_bytes()).hexdigest() != saved_source["sha256"]:
    raise ValueError("Saved helper does not match its provenance record.")
spec = importlib.util.spec_from_file_location("chapter18_saved_studio", source_file)
studio = importlib.util.module_from_spec(spec); spec.loader.exec_module(studio)
data = studio.load_data(DATA_DIR)
choices = [(row.derivative_id, int(i)) for i, row in data["manifest"].iterrows() if row.split == "validation"]
print("Exploration is restricted to validation rows:", len(choices))

## One tool call, several complementary views
Grad-CAM uses the eyes-closed logit for target 1 and its negative for target 0.
The chosen convolutional map is 32×32; enlarging it to 64×64 does not create more precise localization.
Occlusion replaces each 16×16 tile with pixel value 128, approximately the display midpoint, **not** the training mean or absent physiology.
Its signed output change always refers to P(eyes closed), independent of the heatmap target selector.
Retrieval compares trained convolutional features with **training-gallery** images only; similarity is not clinical equivalence.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
def review(index, target):
    record = studio.inspect_example(DATA_DIR, index, target)
    print(json.dumps(record, indent=2))
display(widgets.interactive(review, {"manual": True},
    index=widgets.Dropdown(options=choices, description="Example"),
    target=widgets.Dropdown(options=[("Open logit",0),("Closed logit",1)],value=1,description="CAM target")))

## Challenge the explanation on validation data
A heatmap is a probe, not a physiological source map. Compare it with the replacement map,
then inspect a randomized-parameter control. Similar-looking maps can be misleading.
Quantify the comparison, but do not invent a universal pass/fail correlation threshold.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
index = choices[0][1]
pixel = data["pixels"][index]
trained = tf.keras.models.load_model(DATA_DIR / "model.keras")
randomized = studio.make_model(seed=1818)  # control model; no labels fitted
maps = [studio.gradcam(m, pixel, target=1) for m in [trained, randomized]]
fig, axes = plt.subplots(1,2,figsize=(7,3))
for ax, heat, title in zip(axes,maps,["Trained parameters","Random parameters"]):
    ax.imshow(heat,cmap="magma",vmin=0,vmax=1); ax.set_title(title)
fig.savefig(DATA_DIR / "cam_randomization_control.png", dpi=150, bbox_inches="tight")
plt.show()
correlation = (float(np.corrcoef(maps[0].ravel(),maps[1].ravel())[0,1])
               if all(np.std(z)>0 for z in maps) else None)
studio.write_json(DATA_DIR / "cam_control.json", {"validation_derivative": choices[0][0],
    "pearson_correlation": correlation, "control": "fresh random parameters; not label-shuffled retraining"})
print("Correlation (None for constant maps):",correlation)

## Add a structured review interface, without a paid API
The record below is a **hand-authored schema example**, not a VLM-generated interpretation.
A future locally run or hosted VLM could be asked to use the same allowed categories, panel IDs, and explicit review boundary.
Do not send private data to an external endpoint without the appropriate authorization.
This exercise validates fields and references; it cannot validate the truth of a model's explanation.

In [ ]:
derivative_id = choices[0][0]
panel_ids = ["input", "gradcam", "replacement", "training_gallery"]
response = {"derivative_id": derivative_id, "category": "defer",
            "evidence_panels": ["input", "replacement"], "review_status": "pending"}
assert studio.check_review_response(response, derivative_id, panel_ids)
studio.write_json(DATA_DIR / "review_schema_example.json", response)
try:
    studio.check_review_response({**response, "evidence_panels": ["nonexistent_panel"]}, derivative_id, panel_ids)
except ValueError as error:
    print("Invalid evidence reference rejected:",error)

## Explain and save
1. Trace one displayed prediction back to its model hash, pixels, numeric PSD, waveform, and source EDF.
2. Explain why a bright heatmap and nearest neighbor do not validate a biomarker.
3. State the occlusion replacement and why it may create an unfamiliar input.
4. Distinguish schema checking from checking the truth of a response.
5. Save an inspection record, randomized-map comparison, figure, and human reflection.
**You have built a useful research teaching tool:** it makes model behavior inspectable and records what was reviewed.
It does not remove artifacts, diagnose participants, call a VLM, or establish clinical readiness.